## Causal attention msm

In [2]:
import numpy as np
import torch
import math
import torch.nn as nn

In [3]:
class CausalAttentionV2(nn.Module):
    def __init__(self, dim_in, dim_out, context_length, droput_percentage=0.0, qkv_bias=False):
        super().__init__()

        self.dim_in = dim_in
        self.dim_out = dim_out
        self.context_length = context_length

        self.drop_out = nn.Dropout(p=droput_percentage)

        self.weight_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_key = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)

        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, inputs):
        queries = self.weight_query(inputs)
        keys = self.weight_key(inputs)
        values = self.weight_value(inputs)

        attention_scores = queries @ keys.T

        scaled_weights = attention_scores / math.sqrt(keys.shape[-1])

        causal_attention_scores = attention_scores.masked_fill(self.mask.bool(), -torch.inf)

        causal_scaled_weights = causal_attention_scores / math.sqrt(keys.shape[-1])

        causal_attention_weights = torch.softmax(causal_scaled_weights, dim=1)

        # apply dropout 
        causal_attention_weights = self.drop_out(causal_attention_weights)

        context_vector = causal_attention_weights @ values

        return context_vector

In [4]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

In [5]:
dim_in = inputs.shape[-1]
dim_out = 2
context_length = inputs.shape[0]

torch.manual_seed(123)

causal_attention = CausalAttentionV2(dim_in=dim_in, dim_out=dim_out,context_length=context_length)

context_vector = causal_attention(inputs)
context_vector

tensor([[-0.4519,  0.2216],
        [-0.5874,  0.0058],
        [-0.6300, -0.0632],
        [-0.5675, -0.0843],
        [-0.5526, -0.0981],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)